# Select Link Analysis and Summary

In [27]:
import os
import pandas as pd
import math
from IPython.display import display
import numpy as py
import ast
import warnings
# warnings.filterwarnings(action='once')
warnings.filterwarnings('ignore')
import openpyxl
import time
import datetime

import inro.modeller as _m
import inro.emme.database.emmebank as _eb

## User Defined Inputs

User needs to define the following options and inputs to perform the select link analysis and summary before running the process.

#### Choose ABM version of the scenario ("ABM2" or "ABM2+")

In [28]:
# enter "ABM2" or "ABM2+"
abm = "ABM2+"

#### Define Select Link Query and suffix

In [29]:
# enter True if desire to run select link analysis, otherwise enter False
run_select_link = True

# enter select link query expression and corresponding suffix (suffix length is up to 6 characters)
# select link query format is ("query expression", "suffix")
# if multiple select link queries, the format is ("query expression 1", "suffix 1"), ("query expression 2", "suffix 2")
sl_exp_suffix = [("i=3860 or j=3860", "z3860")]

#### Do you need to run select link flow summary or demand summary?

In [30]:
# enter True if desire to run select link flow summary, otherwise enter False
sl_flow_summary = True

# enter True if desire to run select link demand matrix summary, otherwise enter False
sl_demand_summary = True

#### Do you need to delete existing select link results from EMME database (network volume and demand matrices)?

In [31]:
# Enter True only if you want to delete the results from database. They CANNOT be recovered. 
delete_option = True

# Enter select link suffix to be deleted
query_to_delete = ["z3190", "z2839", "z4522", "z1440", "z4995", "z1935", "z3012", "z114"]


#### Do you need to expand the size of database? (Default to False)

In [32]:
# enter True only if the scenario has more than 4 select link analysis done. 
# Note the database expansion process could take more than 2 hours

expand_option = False

### based on number of selected regions in query set this number. 40000000 is the default for 4 regions
required_dimensions = 40000000

## Initialization

In [33]:
# program timer initiated
start_time_program = time.time()

In [34]:
modeller = _m.Modeller()
desktop = modeller.desktop
emmebank = modeller.emmebank

project_dir = os.path.dirname(_m.Modeller().desktop.project.path)
main_dir = os.path.dirname(project_dir)
database_dir = os.path.join(project_dir, "Database")
output_dir = os.path.join(main_dir, "output", "sl")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
period_dir = os.path.join(output_dir, "by_period")
if not os.path.exists(period_dir):    
    os.makedirs(period_dir)

In [35]:
# EMME tools
change_db_dimension = modeller.tool("inro.emme.data.database.change_database_dimensions")
import_attr_values = modeller.tool("inro.emme.data.network.import_attribute_values")
create_attr = modeller.tool("inro.emme.data.extra_attribute.create_extra_attribute")
copy_attr = modeller.tool("inro.emme.data.network.copy_attribute")
traffic_assign  = modeller.tool("sandag.assignment.traffic_assignment")
delete_matrix = modeller.tool("inro.emme.data.matrix.delete_matrix")
export_to_omx = modeller.tool("inro.emme.data.matrix.export_to_omx")
compute_matrix = modeller.tool("inro.emme.matrix_calculation.matrix_calculator")  
create_matrix = modeller.tool("inro.emme.data.matrix.create_matrix")

In [36]:
scenario_id = 100
periods = ["EA", "AM", "MD", "PM", "EV"]
period_ids = list(enumerate(periods, start=int(scenario_id) + 1))

In [37]:
# emmebank and networks
main_emmebank = _eb.Emmebank(os.path.join(project_dir, "Database", "emmebank"))
base_scenario = emmebank.scenario(scenario_id)

base_network = base_scenario.get_network()

In [38]:
# define vehicle class by abm version
if abm == "ABM2":
    auto_classes = [
        'SOVGPL', 'SOVTOLLL', 'HOV2HOVL', 'HOV2TOLLL', 'HOV3HOVL', 'HOV3TOLLL',
        'SOVGPM', 'SOVTOLLM', 'HOV2HOVM', 'HOV2TOLLM', 'HOV3HOVM', 'HOV3TOLLM',
        'SOVGPH', 'SOVTOLLH', 'HOV2HOVH', 'HOV2TOLLH', 'HOV3HOVH', 'HOV3TOLLH'
    ]
    truck_classes = [
        'TRKHGP', 'TRKHTOLL',
        'TRKLGP', 'TRKLTOLL', 
        'TRKMGP','TRKMTOLL'
    ]
    truck_classes_pce = [
        ('TRKHGP', 2.5), ('TRKHTOLL', 2.5),
        ('TRKLGP', 1.3), ('TRKLTOLL', 1.3),
        ('TRKMGP', 1.5), ('TRKMTOLL', 1.5),
    ]
elif abm == "ABM2+":
    auto_classes = [
        'SOV_NT_L', 'SOV_TR_L', 'HOV2_L', 'HOV3_L',
        'SOV_NT_M', 'SOV_TR_M', 'HOV2_M', 'HOV3_M',
        'SOV_NT_H', 'SOV_TR_H', 'HOV2_H', 'HOV3_H'
    ]
    truck_classes = [
        'TRK_L', 'TRK_M', 'TRK_H'
    ]
    truck_classes_pce = [
        ('TRK_L', 1.3),
        ('TRK_M', 1.5),
        ('TRK_H', 2.5)
    ]
else:
    raise ValueError("Please enter correct model version.")

In [39]:
# populate select link query by expression and suffix specified by user
select_link_query = []
suffix_list = []
for exp, suffix in sl_exp_suffix:
    suffix = str(suffix)
    query_string = '{"expression": "%s", "suffix": "%s", "threshold": 1}' % (exp, suffix)
    query = ast.literal_eval(query_string)
    select_link_query.append(query)
    suffix_list.append(suffix)
select_link_query

[{'expression': 'i=2119 or j=2119', 'suffix': 'z2119', 'threshold': 1}]

## Helper Function

In [40]:
# function to delete extra attributes for specified select link queries on network
def delete_extra_attribute(suffixes):
    for suffix in suffixes:
        suffix = str(suffix)
        auto_attrs = ["@sl_" + name.lower() + "_" + suffix
                  for name in auto_classes]

        truck_attrs = ["@sl_" + name.lower() + "_" + suffix
                       for name in truck_classes]

        for number, period, in enumerate(periods, start=scenario_id + 1):
            scenario = main_emmebank.scenario(number)    
            for name in auto_attrs: 
                if scenario.extra_attribute(name):
                    scenario.delete_extra_attribute(name)            
            for name in truck_attrs: 
                if scenario.extra_attribute(name):
                    scenario.delete_extra_attribute(name)

            if scenario.extra_attribute("@slink_"+ suffix):
                scenario.delete_extra_attribute("@slink_"+ suffix) 

In [41]:
# function to delete demand matrices for specified select link queries in database
def delete_matrices(suffixes):
    matrix_id_list = []
    matrix_id_name = []
    for suffix in suffixes: 
        suffix = str(suffix)
        for matrix in main_emmebank.matrices():
            _suffix = " " + suffix
            if (_suffix in matrix.description) and ("Selected demand" in matrix.description):
                matrix_id_list.append(matrix.id)

    for i in range(0, (len(matrix_id_list))):
            delete_matrix(matrix_id_list[i])

In [42]:
# function to expand EMME database dimension
def expand_database_dimension(required_dimensions):
    new_dimensions = emmebank.dimensions
    if new_dimensions["extra_attribute_values"] < required_dimensions :
        new_dimensions["extra_attribute_values"] = required_dimensions
        change_db_dimension(emmebank_dimensions=new_dimensions,
                  keep_backup=True)

In [43]:
# select link analysis
def run_select_link_analysis(select_link_query):
    msa_iteration = 4
    relative_gap = 0.0005
    max_assign_iterations = 100
    num_processors = "MAX-1"
    select_link = select_link_query

    for number, period in period_ids:
        period_scenario = main_emmebank.scenario(number)
        traffic_assign(period, msa_iteration, relative_gap, max_assign_iterations,
                       num_processors, period_scenario, select_link)

In [44]:
# export the selec link flow by time period and also calculate daily flow on each link
def export_period_flow(suffix):  
    suffix = str(suffix)
    auto_attrs = ["@sl_" + name.lower() + "_" + suffix
                  for name in auto_classes]

    truck_attrs = [("@sl_" + name.lower() + "_" + suffix, pce)
                   for name, pce in truck_classes_pce]

    for number, period, in enumerate(periods, start=scenario_id + 1):
        scenario = emmebank.scenario(number)
        network = scenario.get_network()
    
        tcov_id_list =[]
        sl_flow_list =[]
        non_pce_flow_list =[]
    
        for link in network.links():
            tcov_id = link["@tcov_id"]
            tcov_id_list.append(tcov_id)
    
            sl_flow = sum(link[att] for att in auto_attrs)
            sl_flow += sum(link[att] / pce for att, pce in truck_attrs)
            sl_flow_list.append(sl_flow)
    
            non_pce_flow = link["@non_pce_flow"]
            non_pce_flow_list.append(non_pce_flow)
        
        list_of_fields = list(zip(tcov_id_list, sl_flow_list, non_pce_flow_list))  
        df_link = pd.DataFrame(list_of_fields, columns = ['tcov_id', 'sl_flow', 'non_pce_flow']) 
    
        df_link.to_csv(os.path.join(period_dir, ("loadselk_" + suffix + "_" + period + ".csv")), header = True, index= False)


In [45]:
def cal_daily_flow(suffix):
    suffix = str(suffix)
    auto_attrs = ["@sl_" + name.lower() + "_" + suffix
                  for name in auto_classes]

    truck_attrs = [("@sl_" + name.lower() + "_" + suffix, pce)
                   for name, pce in truck_classes_pce]
    
    daily_tcov_id_list =[]
    daily_sl_flow_list =[]
    daily_non_pce_flow_list =[]
    link_length_list =[]

    for number, period, in enumerate(periods, start=scenario_id + 1):
        scenario = emmebank.scenario(number)
        network = scenario.get_network()
        for link in network.links():
            tcov_id = link["@tcov_id"]
            daily_tcov_id_list.append(tcov_id)
    
            sl_flow = sum(link[att] for att in auto_attrs)
            sl_flow += sum(link[att] / pce for att, pce in truck_attrs)
            daily_sl_flow_list.append(sl_flow)
    
            non_pce_flow = link["@non_pce_flow"]
            daily_non_pce_flow_list.append(non_pce_flow)
            
            link_length = link["length"]
            link_length_list.append(link_length)
    
    daily_list_of_fields = list(zip(daily_tcov_id_list,link_length_list, daily_sl_flow_list, daily_non_pce_flow_list))  
    daily_df_link = pd.DataFrame(daily_list_of_fields, columns = ['tcov_id', 'length', 'sl_flow', 'non_pce_flow'])  
    
    # Filter out transit aux links tcov_id = 0 
    daily_df_link = daily_df_link.loc[daily_df_link['tcov_id'] != 0]
    
    #aggregate to daily value
    daily_df_link = daily_df_link.groupby(['tcov_id'], as_index=False).sum()    

    #identify AB/BA flow
    daily_df_link['dir'] = "AB"    
    daily_df_link.loc[daily_df_link['tcov_id'] < 0, 'dir'] = "BA"
    daily_df_link['tcov_id_'] = abs(daily_df_link['tcov_id'])
    
    # obtain link length
    daily_df_link_len = daily_df_link.pivot_table(index=['tcov_id_'],columns='dir',values='length',fill_value=0)
    daily_df_link_len.columns = ['AB_length', 'BA_length']
    daily_df_link_len['length'] = daily_df_link_len['AB_length'] / len(periods) 
    daily_df_link_len = daily_df_link_len[['length']]
    
    #spread AB/BA selected link flow
    daily_df_link_sl = daily_df_link.pivot_table(index=['tcov_id_'],columns='dir',values='sl_flow',fill_value=0)
    daily_df_link_sl.columns = ['AB_SL_Flow', 'BA_SL_Flow']
    daily_df_link_sl['Total_SL'] = daily_df_link_sl['AB_SL_Flow'] + daily_df_link_sl['BA_SL_Flow']
    
    #spread AB/BA total link flow
    daily_df_link_tot = daily_df_link.pivot_table(index=['tcov_id_'],columns='dir',values='non_pce_flow',fill_value=0)
    daily_df_link_tot.columns = ['AB_Tot_Flow', 'BA_Tot_Flow']
    daily_df_link_tot['Total_Tot'] = daily_df_link_tot['AB_Tot_Flow'] + daily_df_link_tot['BA_Tot_Flow']
    
    #merge selected and total link flow and calculate percentage
    daily_df_link_merge = pd.concat([daily_df_link_len, daily_df_link_sl, daily_df_link_tot], axis=1, join='inner')
    # link percentage is percent of select link volume on each link out of all selected volume
    daily_df_link_merge['pct'] = daily_df_link_merge['Total_SL'] / max(daily_df_link_merge['Total_SL'])
    
    return daily_df_link_merge

In [46]:
# function to delete demand matrices for specified select link queries in database
def export_select_link_demand_by_period(suffix):
    for period in periods:
        matrix_id_list = []
        matrix_id_name = []
        
        suffix = str(suffix)
        for matrix in main_emmebank.matrices():
            _suffix = "_" + suffix
            _period = "_" + period
            if (_suffix in matrix.name) and ("SELDEM" in matrix.name) and (_period in matrix.name):
                matrix_id_list.append(matrix.id)
        
        omx_file = os.path.join(period_dir, "select_link_demand_" + suffix + "_%s.omx" % period)        
        export_to_omx(matrices=matrix_id_list, 
                      export_file=omx_file,
                      append_to_file=False)

In [47]:
def calculate_daily_select_link_demand(suffix):
    # create new matrices to store daily select link demand
    auto_day_attrs = ["SELDEM_DAY_" + name.upper() + "_" + suffix
                      for name in auto_classes]
    truck_day_attrs = ["SELDEM_DAY_" + name.upper() + "_" + suffix
                    for name in truck_classes]
    day_attrs = auto_day_attrs + truck_day_attrs    
    
    day_matrix_ID = []
    for id in range(1, len(day_attrs)+1):
        day_matrix_ID.append('mf' + str(id))    
    list_of_daily_matrices = list(zip(day_matrix_ID, day_attrs, day_attrs))   
    
    for matrix in list_of_daily_matrices:
        new_mat = create_matrix(matrix_id=matrix[0],                                
                                matrix_name=matrix[1],
                                matrix_description=matrix[2],
                                default_value=0,
                                overwrite =1)
        
    # aggregate select link demand by period into daily demand
    _suffix = "_" + str(suffix)
    for matrix in list_of_daily_matrices:
        proName = matrix[1].split("_", 2)
#         proName_2 = (matrix[1].split("_", 2)[2]).replace(_suffix, '')
        proTotName = ""
        for P in periods:
            proTotName = proTotName + proName[0] + "_" + P + "_" + proName[2]
            if P != "EV":
                proTotName = proTotName + "+"
        spec = {"expression": "%s" % proTotName,
                "result": matrix[0],
                "constraint": {
                    "by_value": None,
                    "by_zone": None
                    },
                "aggregation": {
                    "origins": None,
                    "destinations": None
                    },
                "type": "MATRIX_CALCULATION"
                }               
        report = compute_matrix([spec])
        
    # export daily demand to omx
    day_omx_file = os.path.join(output_dir, "select_link_demand_" + suffix + "_daily.omx")        
    export_to_omx(matrices=day_matrix_ID,
                  export_file=day_omx_file,
                  append_to_file=False) 
    
    # delete daily demand from emme databank after exporting
    for matrix in list_of_daily_matrices:
        mt=main_emmebank.matrix(matrix[0])
        try:
            delete_matrix(matrix=mt) 
        except:
            pass

## Select Link Analysis

#### Delete Redundant Extra Attributes and Demand Matrices from Previous Select Link Analysis

In [48]:
if delete_option == True:
    delete_extra_attribute(query_to_delete)
    delete_matrices(query_to_delete)

#### Expand Database Dimension

In [49]:
if expand_option == True:
    expand_database_dimension(required_dimensions)

#### Run Select Link Analysis (run time is about 2-3 hours)

In [50]:
if run_select_link == True:
    run_select_link_analysis(select_link_query)

#### Summarize Select Link Analysis Results

In [51]:
if sl_flow_summary == True:
    for suffix in suffix_list:
        daily_df_link = cal_daily_flow(suffix)
        # export daily select link flow
        daily_df_link.to_csv(os.path.join(output_dir, ("Daily_loadselk_" + suffix + ".csv")), header = True, index= True)
        # export select link flow by period
        export_period_flow(suffix)

#### Aggregate Select Link Demand

In [52]:
if sl_demand_summary == True:
    for suffix in suffix_list:
        export_select_link_demand_by_period(suffix)
        calculate_daily_select_link_demand(suffix)

In [53]:
print "Finished Select Link Analysis in %5.2f mins" % ((time.time() - start_time_program)/60.0)

Finished Select Link Analysis in 115.91 mins
